# 2. Estimate the sidelobe contribution

Convolve the native VLA reference maps with the real and Gaussian beam responses. Reproject their difference and assemble the FAST map using the 32 pointing tables. Beam arrays are used with their stored centering; no CRPIX shift is applied.

In [ ]:
import os
import numpy as np
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from scipy.ndimage import shift, zoom
from scipy.signal import fftconvolve
import matplotlib.pyplot as plt

def get_pixscale_arcsec_from_header(header):
    w = WCS(header).celestial
    pixscale = u.Quantity(w.proj_plane_pixel_scales(), u.deg)
    pix_x = abs(pixscale[0].to(u.arcsec).value)
    pix_y = abs(pixscale[1].to(u.arcsec).value)
    return np.sqrt(pix_x * pix_y)

def get_pixel_area_arcmin2(header):
    w = WCS(header).celestial
    pixscale = u.Quantity(w.proj_plane_pixel_scales(), u.deg)
    pix_x = abs(pixscale[0].to(u.arcmin).value)
    pix_y = abs(pixscale[1].to(u.arcmin).value)
    return pix_x * pix_y


In [ ]:
def load_real_and_gauss_on_vla_grid(
    real_file,
    gauss_file,
    vla_pixscale_arcsec
):
    """
    Load real FAST and Gaussian beam arrays with their stored centering,
    resample to VLA pixel scale, and normalize both by the real-beam peak.
    """
    real_raw = fits.getdata(real_file).astype(float)
    gauss_raw = fits.getdata(gauss_file).astype(float)

    real_header = fits.getheader(real_file)
    gauss_header = fits.getheader(gauss_file)

    real_raw = np.nan_to_num(real_raw, nan=0.0)
    gauss_raw = np.nan_to_num(gauss_raw, nan=0.0)

    # Recenter
    real_centered = real_raw
    gauss_centered = gauss_raw

    # Pixel scale conversion
    beam_pixscale_arcsec = get_pixscale_arcsec_from_header(real_header)
    zoom_factor = beam_pixscale_arcsec / vla_pixscale_arcsec

    real_vla = zoom(real_centered, zoom_factor, order=1)
    gauss_vla = zoom(gauss_centered, zoom_factor, order=1)

    real_vla = np.nan_to_num(real_vla, nan=0.0)
    gauss_vla = np.nan_to_num(gauss_vla, nan=0.0)

    # Normalize by real beam peak
    peak = np.nanmax(real_vla)
    if peak <= 0:
        raise ValueError(f'Invalid real-beam peak for {real_file}')

    real_vla = real_vla / peak
    gauss_vla = gauss_vla / peak

    return real_vla, gauss_vla, zoom_factor

In [ ]:
def make_vla_sidelobe_for_one_beam(
    truth_surface,
    truth_header,
    real_kernel_vla,
    gauss_kernel_vla
):
    """
    truth_surface: mJy / arcmin^2 km/s
    kernels: dimensionless response on VLA pixel scale

    output:
        conv_real, conv_gauss, sidelobe all in mJy km/s
    """
    truth_clean = np.nan_to_num(truth_surface, nan=0.0)

    omega_pix = get_pixel_area_arcmin2(truth_header)

    conv_real = fftconvolve(
        truth_clean,
        real_kernel_vla,
        mode='same'
    ) * omega_pix

    conv_gauss = fftconvolve(
        truth_clean,
        gauss_kernel_vla,
        mode='same'
    ) * omega_pix

    sidelobe = conv_real - conv_gauss

    return sidelobe, conv_real, conv_gauss

In [ ]:
beam_dir = './sidelobe-corr/beam-big'
truth_dir = './VLA'
outdir = './sidelobe-result-fits/VLAgrid_sidelobe_test'
os.makedirs(outdir, exist_ok=True)

source_name = 'HCG100'
truth_file = f'{truth_dir}/truth_{source_name}_scaled_VLA_M0_mJyPerArcmin2.fits'

truth = fits.getdata(truth_file)
truth_header = fits.getheader(truth_file)
truth_header['BUNIT'] = 'mJy/beam km/s'

vla_pixscale_arcsec = get_pixscale_arcsec_from_header(truth_header)
omega_pix = get_pixel_area_arcmin2(truth_header)

print('VLA pixscale [arcsec] =', vla_pixscale_arcsec)
print('VLA pixel area [arcmin^2] =', omega_pix)

for ibeam in [1, 17, 18]:
    real_file = f'{beam_dir}/M{ibeam:02d}-beam-kernal.fits'
    gauss_file = f'{beam_dir}/M{ibeam:02d}-beam-clean2.fits'

    real_vla, gauss_vla, zoom_factor = load_real_and_gauss_on_vla_grid(
        real_file,
        gauss_file,
        vla_pixscale_arcsec
    )

    side_vla, conv_real, conv_gauss = make_vla_sidelobe_for_one_beam(
        truth,
        truth_header,
        real_vla,
        gauss_vla
    )

    print(f'M{ibeam:02d}')
    print('  zoom_factor =', zoom_factor)
    print('  kernel shape =', real_vla.shape)
    print('  real peak =', np.nanmax(real_vla))
    print('  gauss peak =', np.nanmax(gauss_vla))
    print('  side min/max =', np.nanmin(side_vla), np.nanmax(side_vla))

    fits.writeto(
        f'{outdir}/{source_name}_M{ibeam:02d}_conv_real_VLAgrid.fits',
        conv_real.astype('float32'),
        truth_header,
        overwrite=True
    )

    fits.writeto(
        f'{outdir}/{source_name}_M{ibeam:02d}_conv_gauss_VLAgrid.fits',
        conv_gauss.astype('float32'),
        truth_header,
        overwrite=True
    )

    fits.writeto(
        f'{outdir}/{source_name}_M{ibeam:02d}_sidelobe_VLAgrid.fits',
        side_vla.astype('float32'),
        truth_header,
        overwrite=True
    )

In [ ]:
for ibeam in [1, 17, 18]:
    side = fits.getdata(f'{outdir}/{source_name}_M{ibeam:02d}_sidelobe_VLAgrid.fits')

    plt.figure(figsize=(6, 5))
    vmax = np.nanpercentile(np.abs(side), 99.5)
    plt.imshow(side, origin='lower', vmin=-vmax, vmax=vmax)
    plt.title(f'{source_name} M{ibeam:02d} sidelobe on VLA grid')
    plt.colorbar(label='mJy km/s')
    plt.show()

In [ ]:
plt.subplot(1,2,1)
plt.imshow(truth, origin='lower')
plt.title('truth')
plt.colorbar()

plt.subplot(1,2,2)
plt.imshow(conv_gauss, origin='lower')
plt.title('truth convolved with Gaussian beam')
plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
source_names = ['HCG100', 'NGC7810', 'MRK935', 'AGC105092']

outdir_all = './sidelobe-result-fits/VLAgrid_sidelobe_by_source_beam'
os.makedirs(outdir_all, exist_ok=True)

for source_name in source_names:
    print('Processing source:', source_name)

    truth_file = f'{truth_dir}/truth_{source_name}_scaled_VLA_M0_mJyPerArcmin2.fits'
    truth = fits.getdata(truth_file)
    truth_header = fits.getheader(truth_file)
    truth_header['BUNIT'] = 'mJy/beam km/s'

    vla_pixscale_arcsec = get_pixscale_arcsec_from_header(truth_header)

    for ibeam in range(1, 20):
        real_file = f'{beam_dir}/M{ibeam:02d}-beam-kernal.fits'
        gauss_file = f'{beam_dir}/M{ibeam:02d}-beam-clean2.fits'

        real_vla, gauss_vla, zoom_factor = load_real_and_gauss_on_vla_grid(
            real_file,
            gauss_file,
            vla_pixscale_arcsec
        )

        side_vla, conv_real, conv_gauss = make_vla_sidelobe_for_one_beam(
            truth,
            truth_header,
            real_vla,
            gauss_vla
        )

        fits.writeto(
            f'{outdir_all}/sidelobe_{source_name}_M{ibeam:02d}_VLAgrid.fits',
            side_vla.astype('float32'),
            truth_header,
            overwrite=True
        )

    print('Done:', source_name)

In [ ]:
for source_name in source_names:
    side_sum = None

    for ibeam in range(1, 20):
        fname = f'{outdir_all}/sidelobe_{source_name}_M{ibeam:02d}_VLAgrid.fits'
        data = fits.getdata(fname)
        header = fits.getheader(fname)

        if side_sum is None:
            side_sum = np.zeros_like(data)

        side_sum += data

    fits.writeto(
        f'{outdir_all}/diagnostic_sum19_sidelobe_{source_name}_VLAgrid.fits',
        side_sum.astype('float32'),
        header,
        overwrite=True
    )

## VLA grid → FAST grid

In [ ]:
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS
from reproject import reproject_interp
import astropy.units as u
import numpy as np
import os

def regrid_vla_to_fast(map_vla, vla_header, fast_header_2d):
    """
    Regrid VLA-grid sidelobe map to FAST grid.
    No convolution, only interpolation/reprojection.
    """
    vla_wcs = WCS(vla_header).celestial
    fast_wcs = WCS(fast_header_2d).celestial

    ny_fast = fast_header_2d['NAXIS2']
    nx_fast = fast_header_2d['NAXIS1']

    map_fast, footprint = reproject_interp(
        (map_vla, vla_wcs),
        fast_wcs,
        shape_out=(ny_fast, nx_fast)
    )

    map_fast[footprint == 0] = np.nan
    map_fast = np.nan_to_num(map_fast, nan=0.0)

    return map_fast

In [ ]:
def assemble_fast_sidelobe_from_beams(
    source_name,
    vla_sidelobe_dir,
    fast_header_2d,
    pointing_list,
    pointing_dir='./pointing-check',
    save_regridded_beams=False,
    outdir_fast_beams=None,
):
    """
    For one source:
    1. Regrid M01-M19 VLA-grid sidelobe maps to FAST grid.
    2. Use pointing/beam assignment to assemble final FAST-grid sidelobe map.
    """

    ny_fast = fast_header_2d['NAXIS2']
    nx_fast = fast_header_2d['NAXIS1']
    fast_wcs = WCS(fast_header_2d).celestial

    # First regrid each beam map to FAST grid
    side_fast_by_beam = {}

    if save_regridded_beams and outdir_fast_beams is not None:
        os.makedirs(outdir_fast_beams, exist_ok=True)

    for ibeam in range(1, 20):
        beam_name = f'M{ibeam:02d}'

        vla_file = f'{vla_sidelobe_dir}/sidelobe_{source_name}_{beam_name}_VLAgrid.fits'

        side_vla = fits.getdata(vla_file)
        vla_header = fits.getheader(vla_file)

        side_fast = regrid_vla_to_fast(
            side_vla,
            vla_header,
            fast_header_2d
        )

        side_fast_by_beam[ibeam] = side_fast

        if save_regridded_beams and outdir_fast_beams is not None:
            fits.writeto(
                f'{outdir_fast_beams}/sidelobe_{source_name}_{beam_name}_FASTgrid_full.fits',
                side_fast.astype('float32'),
                fast_header_2d,
                overwrite=True
            )

    # Then assemble according to actual FAST beam positions
    side_sum = np.zeros((ny_fast, nx_fast), dtype=float)
    weight = np.zeros((ny_fast, nx_fast), dtype=float)

    for ipoint in pointing_list:
        data = np.loadtxt(f'{pointing_dir}/{ipoint}_radec.list')

        ra_beam = data[:, 0]
        dec_beam = data[:, 1]

        for i_beam in range(len(ra_beam)):
            ibeam = i_beam + 1

            coord = SkyCoord(
                ICRS(
                    ra=ra_beam[i_beam] * u.deg,
                    dec=dec_beam[i_beam] * u.deg
                )
            )

            xpix, ypix = fast_wcs.world_to_pixel(coord)
            ix = int(np.round(xpix))
            iy = int(np.round(ypix))

            if ix < 0 or ix >= nx_fast or iy < 0 or iy >= ny_fast:
                continue

            side_sum[iy, ix] += side_fast_by_beam[ibeam][iy, ix]
            weight[iy, ix] += 1.0

    source_side_fast = np.zeros_like(side_sum)
    valid = weight > 0
    source_side_fast[valid] = side_sum[valid] / weight[valid]

    return source_side_fast, weight

In [ ]:
source_names = ['HCG100', 'NGC7810', 'MRK935', 'AGC105092']

pointing_list = [
    'HCG100_p1','HCG100_p2','HCG100_p3','HCG100_p4',
    'HCG100_p5','HCG100_p6','HCG100_p7','HCG100_p8',
    'HCG100_p9','HCG100_p10','HCG100_p11','HCG100_p12',
    'HCG100_p13','HCG100_p14','HCG100_p15','HCG100_p16',
    'HCG100_p17','HCG100_p18','HCG100_p19','HCG100_p20',
    'HCG100_p21','HCG100_p22','HCG100_p23','HCG100_p24',
    'HCG100_p25','HCG100_p26','HCG100_p27','HCG100_p28',
    'HCG100_p29','HCG100_p30','HCG100_p31','HCG100_p32'
]

vla_sidelobe_dir = './sidelobe-result-fits/VLAgrid_sidelobe_by_source_beam'
outdir = './sidelobe-result-fits/VLAgrid_method_FASTgrid_M0'
os.makedirs(outdir, exist_ok=True)

# 用 FAST moment0 header 或 FAST cube 的某一层 header
fast_m0_file = './moments/moment0_5200_5600_2fov_mJykmsBeam.fits'
fast_header_2d = fits.getheader(fast_m0_file)

sidelobe_total_fast = None

for source_name in source_names:
    print('Assembling:', source_name)

    source_side_fast, weight = assemble_fast_sidelobe_from_beams(
        source_name,
        vla_sidelobe_dir,
        fast_header_2d,
        pointing_list,
        pointing_dir='./pointing-check',
        save_regridded_beams=False
    )

    fits.writeto(
        f'{outdir}/sidelobe_{source_name}_M0_FASTgrid.fits',
        source_side_fast.astype('float32'),
        fast_header_2d,
        overwrite=True
    )

    fits.writeto(
        f'{outdir}/weight_{source_name}_FASTgrid.fits',
        weight.astype('float32'),
        fast_header_2d,
        overwrite=True
    )

    if sidelobe_total_fast is None:
        sidelobe_total_fast = np.zeros_like(source_side_fast)

    sidelobe_total_fast += source_side_fast

fits.writeto(
    f'{outdir}/sidelobe_total_M0_FASTgrid.fits',
    sidelobe_total_fast.astype('float32'),
    fast_header_2d,
    overwrite=True
)

print('Done.')

In [ ]:
import matplotlib.pyplot as plt

for source_name in source_names:
    data = fits.getdata(f'{outdir}/sidelobe_{source_name}_M0_FASTgrid.fits')

    vmax = np.nanpercentile(np.abs(data), 99.5)

    plt.figure(figsize=(5, 5))
    plt.imshow(data, origin='lower', vmin=-vmax, vmax=vmax)
    plt.title(f'{source_name} sidelobe M0 on FAST grid')
    plt.colorbar(label='mJy km/s')
    plt.show()

In [ ]:
total = fits.getdata(f'{outdir}/sidelobe_total_M0_FASTgrid.fits')

vmax = np.nanpercentile(np.abs(total), 99.5)

plt.figure(figsize=(6, 5))
plt.imshow(total, origin='lower', vmin=-vmax, vmax=vmax)
plt.title('Total sidelobe M0 on FAST grid')
plt.colorbar(label='mJy km/s')
plt.show()